# QML

### Creer un modèle de classification quantique
 - Pour classifier l'etat d'un cancer (Malignant ou Benign)
 - Pour distinguer un chien, d'un chat.

## Classifier l'etat d'un cancer (Malignant ou Benign)

In [67]:
import numpy as np
import random

#Avant tout, il faut recuperer notre dataset

# importer les donnees
data_bc= np.genfromtxt('./datasets/breast-cancer-wisconsin-data_data.csv',
  skip_header=1,delimiter=',',encoding=None,dtype=None,
  converters={1: lambda x: 0 if x == 'M' else 1})

nbr_features=4#len(data_bc[0])-2

# les separer en deux classes
# 0: malignant, 1: benign

data_bc_class=[[],[]]
for i in range(len(data_bc)):
  _data_bc=[]
  for j in range(nbr_features+1):
    _data_bc.append(data_bc[i][j+1])
  if data_bc[i][1] == 0:
    data_bc_class[0].append(_data_bc)
  else:
    data_bc_class[1].append(_data_bc)

# le nombre d'items de chaque classe
nbr_class0=len(data_bc_class[0])
nbr_class1=len(data_bc_class[1])

# partitionner les donnees en donnees d'entrainement et de test
_train0=data_bc_class[0][:int(nbr_class0/2)]
_test0=data_bc_class[0][int(nbr_class0/2):]
_train1=data_bc_class[1][int(nbr_class1/2):]
_test1=data_bc_class[1][:int(nbr_class1/2)]

d_train=_train0+_train1
d_test=_test0+_test1

# les donnees actuelles sont en ordre
# il faut les melanger
# pour que le classifieur ne soit pas influence par l'ordre
# de presentation des donnees
random.seed(49)
random.shuffle(d_train)
random.shuffle(d_test)

# Extract the second and third elements from each sublist
x_train = np.array([item[1:] for item in d_train])
y_train=np.array([item[:1] for item in d_train])
x_test=np.array([item[1:] for item in d_test])
y_test=np.array([item[:1] for item in d_test])

print('nbr of items: ' + str(len(data_bc)))
print('nbr of features: ' + str(nbr_features))
print('nbr of classes: 2')
print('nbr of class 0: ' + str(nbr_class0))
print('nbr of class 1: ' + str(nbr_class1))
print('nbr of x_train items: ' + str(len(x_train)))
print('nbr of x_test items: ' + str(len(x_test)))
print('nbr of y_train items: ' + str(len(y_train)))
print('nbr of y_test items: ' + str(len(y_test)))


nbr of items: 569
nbr of features: 4
nbr of classes: 2
nbr of class 0: 212
nbr of class 1: 357
nbr of x_train items: 285
nbr of x_test items: 284
nbr of y_train items: 285
nbr of y_test items: 284


## Creation du circiut parametre

In [68]:
#encodage par angle
from qiskit.circuit import QuantumCircuit, Parameter
from qiskit.circuit.library import TwoLocal

emb_circuit = QuantumCircuit(nbr_features)

params = [Parameter(f'p{str(i)}') for i in range(nbr_features)]  

for i in range(nbr_features):
    emb_circuit.rx(params[i], i)

ansatz = TwoLocal(nbr_features, ['rz', 'rx'], 'cx', 'linear', reps=2, parameter_prefix='w')

qc= emb_circuit.compose(ansatz)

## Creation du Reseau de Neurone Quantique

In [69]:
from qiskit_algorithms.optimizers import COBYLA
from qiskit_machine_learning.neural_networks import SamplerQNN

def interpret(x):
    return '{:b}'.format(x).count('1') % 2

qnn = SamplerQNN(circuit=qc,
                         input_params=emb_circuit.parameters, 
                         weight_params=ansatz.parameters, 
                         interpret=interpret, 
                         output_shape=2)

/tmp/ipykernel_306820/750675033.py:7: DeprecationWarning: V1 Primitives are deprecated as of qiskit-machine-learning 0.8.0 and will be removed no sooner than 4 months after the release date. Use V2 primitives for continued compatibility and support.
  qnn = SamplerQNN(circuit=qc,


### Observons de quoi est capable notre reseau de neuronne

In [70]:
# Initialisons les poids aleatoirement
np.random.seed(49)
weight_al=np.random.rand(ansatz.num_parameters)

# Testons sa capacite de classification
res = qnn.forward(x_train[0], weight_al)
print("Probabilite d'etre Malignant :"+ str(res[0][0]))
print("Probabilite d'etre Benign :"+ str(res[0][1]))

Probabilite d'etre Malignant :0.6175639845579338
Probabilite d'etre Benign :0.38243601544206535


## Entrainement du classifieur

In [71]:
# Optimiseur de parametres
num_iter = 20
optimizer = COBYLA(maxiter=num_iter)

In [ ]:
# Algorithme d'optimisation
from qiskit_machine_learning.algorithms import NeuralNetworkClassifier

# Convert y_train and y_test to single-dimensional arrays of class labels
y_train_labels = np.array([np.argmax(y) for y in y_train])
y_test_labels = np.array([np.argmax(y) for y in y_test])

nnc = NeuralNetworkClassifier(neural_network=qnn,
  optimizer=optimizer, initial_point=weight_al)

nnc.fit(x_train, y_train_labels)

sc_train=nnc.score(x_train,y_train)
sc_test=nnc.score(x_test,y_test)
print("Score d'entrainement : " + str(sc_train))
print("Score de test : " + str(sc_test))

QiskitMachineLearningError: "Shapes don't match, predict: (285,), target: (285, 1)!"

## Effectuons une prediction

In [ ]:
select_item=random.randint(0,len(data_bc)-1)
_item=data_bc[select_item]
item=[]
for j in range(len(_item)-2):
  item.append(_item[j+2])
print("Item de test : " + str(_item))

pred=nnc.predict(item)
print("Prediction : " + str(pred))

: 

: 

: 

: 

: 

: 

: 